In [1]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pymongo import MongoClient, ASCENDING, UpdateOne, DESCENDING
from datetime import datetime
from dotenv import load_dotenv
import os

In [3]:
load_dotenv("../.env")
MONGO_URI  = os.environ["MONGO_URI"]
DB_NAME    = os.environ["MONGO_DB"]
COLLECTION = os.environ["MONGO_COLLECTION_TIMESERIES"]
COLLECTION_METADATA = os.environ["MONGO_COLLECTION_METADATA"]

client = MongoClient(MONGO_URI)
db = client[DB_NAME]
metadata = db[COLLECTION_METADATA]
collection = db[COLLECTION]

In [8]:
result = collection.delete_many({"timestamp": target_timestamp})
print(f"Deleted {result.deleted_count} documents")

Deleted 6 documents


In [ ]:
result = metadata.find_one()
print(result)

In [ ]:
doc = metadata.find_one(
    {"_id": "Gemini"},
    {"installed_capacity_mw": 1, "_id": 0}
)

In [ ]:
print(doc)

In [4]:
result = collection.find_one()
print(result)

{'_id': ObjectId('6a32ac0a6f24d4b847193bf2'), 'timestamp': datetime.datetime(2026, 1, 1, 0, 0), 'farm_id': 'TOTAL', 'actual_mw': 4287.27, 'predictions': {'knowledge_based_mw': 3753.9300352947234}}


In [ ]:
latest = collection.find(
    {"farm_id": "TOTAL"}
)
for i in latest:
    print(i)

In [ ]:
latest_10 = collection.find(
    {"farm_id": "TOTAL"}
).sort("timestamp", DESCENDING).limit(10)

for doc in latest_10:
    print(doc)

In [ ]:
def insert_timestep(timestamp, farm_predictions: dict, actual_total_mw: float = None):
    """
    farm_predictions example:
    {
        "Borssele_12": {"wind_speed_ms": 8.5, "predictions": {"knowledge_based_mw": 450.2, "decision_tree_mw": 440.1, "neural_network_mw": 455.8}},
        "Gemini": {...},
        ...
    }
    """
    collection = client[DB_NAME]["Timeseries"]

    docs = []
    totals = {}

    for farm_id, data in farm_predictions.items():
        doc = {
            "timestamp": timestamp,
            "farm_id": farm_id,
            "wind_speed_ms": data.get("wind_speed_ms"),
            "predictions": data["predictions"],
        }
        docs.append(doc)

        for model_name, value in data["predictions"].items():
            totals[model_name] = totals.get(model_name, 0) + (value or 0)

    docs.append({
        "timestamp": timestamp,
        "farm_id": "TOTAL",
        "predictions": totals,
        "actual_mw": actual_total_mw,
    })

    collection.insert_many(docs)

In [ ]:
def insert_entsoe_2026(entsoe_csv_path: str):
    df = pd.read_csv(entsoe_csv_path, index_col="Datetime", parse_dates=True)
    df = df[df.index.year == 2026]

    client = MongoClient(MONGO_URI)
    collection = client[DB_NAME][COLLECTION]

    operations = [
        UpdateOne(
            {"timestamp": ts.to_pydatetime(), "farm_id": "TOTAL"},
            {"$set": {"actual_mw": float(row["Generation_MW"]) if pd.notna(row["Generation_MW"]) else None}},
            upsert=True
        )
        for ts, row in df.iterrows()
    ]

    if not operations:
        print("No 2026 rows found in ENTSOE data")
        return

    result = collection.bulk_write(operations, ordered=False)
    print(f"Upserted {result.upserted_count}, modified {result.modified_count} TOTAL documents for 2026")


In [ ]:
insert_entsoe_2026("../datasets/entsoe.csv")

In [ ]:
from pymongo import MongoClient

client = MongoClient(MONGO_URI)
db = client[DB_NAME]

db.create_collection("Timeseries")

# compound unique index so timestamp+farm_id can't be duplicated, and queries stay fast
db["Timeseries"].create_index([("timestamp", 1), ("farm_id", 1)], unique=True)

In [ ]:
df = pd.read_csv("../datasets/entsoe.csv", index_col="Datetime", parse_dates=True)

In [ ]:
"""
{
  "timestamp": "2026-01-01T00:00:00",
  "farm_id": "Borssele_12",
  "weather_data":{
      "wind_speed_ms": 8.5,
      "wind_direction_deg": 1,
      "pressure_hpa": 1,
      "temperature_c": 1
  }
  "predictions": {
    "knowledge_based_mw": 450.2,
    "decision_tree_mw": 440.1,
    "neural_network_mw": 455.8
  }
}
{
  "timestamp": "2026-01-01T00:00:00",
  "farm_id": "TOTAL",
  "predictions": {
    "knowledge_based_mw": 450.2,
    "decision_tree_mw": 440.1,
    "neural_network_mw": 455.8
  },
  "actual_mw": null
}
"""

In [ ]:
{
      '_id': ObjectId('6a350e2edcfa225ba2e347d7'),
      "id": "Borssele_12",
      "name": "Borssele 1&2",
      "installed_capacity_mw": 752,
      "location": {
        "lat": 51.75,
        "lon": 3.25
      },
      "commissioned": "27/11/2020",
      "turbine_type": "SG 8.0-167 DD",
      "platform_height": 116.5,
      "offshore_data_available": true,
      "turbines": 94,
      "nearest_KNMI_station": "Vlissingen",
      "statistical_metrics":{
           'Unnamed: 0': {'max': 350639.0, 'min': 0.0, 'median': 175319.5},
           'u100': {'max': 30.982468, 'min': -18.986801, 'median': 2.60221865},
           'v100': {'max': 28.04042, 'min': -24.91832, 'median': 1.1620636000000002},
           'fsr': {'max': 0.008119814, 'min': 2.425944e-05, 'median': 0.000107233863},
           'Windspeed': {'max': 34.935352, 'min': 0.007264482, 'median': 8.891778500000001},
           'Scaled_Windspeed_(at_116.5m)': {'max': 36.095177, 'min': 0.0075053987, 'median': 9.1866645},
           'Wind_Direction': {'max': 359.99973, 'min': 0.0002746582, 'median': 196.236525},
           'Power_of_SG 8.0-167 DD': {'max': 752.0, 'min': 0.0, 'median': 474.26210000000003},
           'Turn_off': {'max': 1.0, 'min': 0.0, 'median': 1.0},
           'Power': {'max': 752.0, 'min': 0.0, 'median': 474.26210000000003},
      }
}

In [ ]:
client = MongoClient(MONGO_URI)
collection = client[DB_NAME]["Timeseries"]

In [ ]:
results = collection.find(
    {
        "farm_id": "Borssele_12",
    },
    {"_id": 0, "timestamp": 1, "wind_speed_ms": 1}
).sort("timestamp", 1).limit(10)

In [ ]:
print(result)

In [ ]:
client = MongoClient(MONGO_URI)
collection = client[DB_NAME]["Timeseries"]
result = collection.find({"farm_id": "Gemini"}).limit(10)
print(list(result))

In [ ]:
result = collection.find(
    {
        "farm_id": "Gemini",
        "timestamp": {
            "$gte": datetime(2026, 1, 1),
            "$lt": datetime(2026, 2, 1)
        }
    },
    {}
).sort("timestamp", 1)

print(list(result))